# Scénario 11e : `PAY_1_corrigee` à côté de `PAY_1`, dans la prédiction du défaut

*Journal : [`tableau_resultats.md`](tableau_resultats.md). Préréglages repris d'[`essai_jeu_de_base.ipynb`](../comparatif_global/essai_jeu_de_base.ipynb).*

**Question** : une codification de septembre corrigée pour les `PAY_1` = 1, ajoutée **à côté** de `PAY_1` d'origine, aide-t-elle les modèles à prédire le défaut (`dpnm`) ? Et surtout : **que regardent les modèles** quand ils ont les deux colonnes ?

**Construction de `PAY_1_corrigee`** (décisions de l'utilisateur), sur `data/creditcard_pret_ingestion.csv` (niveau 0) :
1. **correction des codifications 1 d'avril à août** (notebook `10`, codification anticipée), en commençant par les plus anciennes : solde du mois suivant nul ou créditeur (codification saine antérieure), encours nul (codification du mois précédent), décisions de l'utilisateur. L'étape des impayés du `10` est retirée : elle reprend la règle fixe des retards, qui n'est pas appliquée (elle est peut-être fausse ; revue en cours dans l'EDA) ;
2. **`PAY_1` différent de 1** : `PAY_1_corrigee` = `PAY_1` ;
3. **`PAY_1` = 1, sain en août** : `PAY_1_corrigee` = la codification d'août (la persistance domine ; aucune entrée en retard n'est détectable, passage 24) ;
4. **`PAY_1` = 1, en retard en août** : les modèles du `11c` (MLP et CatBoost, enregistrés dans `modeles/11c/` avec leurs seuils à 80 % de précision marginale) ; **sortie du retard si les deux l'annoncent** → `PAY_1_corrigee` = -1 (361 sorties sur 366 sont codifiées -1 dans le train) ; sinon **retard maintenu** → 2.

`PAY_1` d'origine est **gardée telle quelle**. `PAY_1_corrigee` est **encodée comme les autres `PAY_n`** d'`essai_jeu_de_base` (nombre ordonné, mis à l'échelle). Les modèles du `11c` ont appris sur `PAY_1`, jamais sur `dpnm` : aucune information sur le défaut n'entre dans `PAY_1_corrigee`.

**Mêmes préréglages qu'`essai_jeu_de_base`** : 30 000 clients, mêmes encodages, 4 modèles, même boucle (précision moyenne, surapprentissage à 7 %), mêmes grilles, même seuil optimisé (au plus 1 bon client signalé pour 1 défaut, à la marge), mêmes 5 plis (`random_state=42`).

**Différences avec `essai_jeu_de_base`** :
- les codifications 1 d'avril à août (`PAY_2` à `PAY_6`, 28 clients) sont corrigées par l'étape 1 ;
- une colonne de plus, `PAY_1_corrigee` ;
- deux variantes : **« Complet + PAY_1 corrigée »** (24 variables) et **« PAY_n seuls + PAY_1 corrigée »** (7) ; la variante « Sans les PAY_n » n'est pas reprise (elle ne contient ni `PAY_1` ni `PAY_1_corrigee`) ;
- les sections « Yeh et Lien » et « comparaison avec les scénarios » sont remplacées par la **comparaison avec `essai_jeu_de_base`** (section 6), pli par pli, sur les mêmes clients ;
- **le plafond (`LIMIT_BAL`) est mis au logarithme** (`log1p`), puis à l'échelle, comme les paiements (décision de l'utilisateur) ; dans `essai_jeu_de_base`, il est seulement mis à l'échelle ;
- probabilités hors pli et colonne construite enregistrées dans le dossier local `sorties/`.

⚠️ **Le test du ML est inclus**, comme dans `essai_jeu_de_base` : aucune conclusion de cet essai ne doit servir à choisir le modèle final ou ses variables.

## 1. Chargement du jeu de base et variables des trois variantes

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

original = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")   # niveau 0 du nettoyage (02_01_nettoyage.ipynb)
assert original["ID"].is_unique and not original.isna().any().any()
y_train = original["dpnm"].reset_index(drop=True)
print(f"Clients : {len(original)} ; colonnes : {original.shape[1]} (ID, 23 variables, dpnm)")
print(f"Défauts : {y_train.sum()} ({y_train.mean() * 100:.2f} %)")

Clients : 30000 ; colonnes : 25 (ID, 23 variables, dpnm)
Défauts : 6636 (22.12 %)


## 1 bis. Correction des codifications 1 d'avril à août (`10`, sans la règle des retards)

Les étapes du notebook `10`, dans l'ordre, des mois les plus anciens aux plus récents. Après correction, il ne doit plus rester aucune codification 1 d'avril à août. `PAY_1` n'est pas touchée.

In [2]:
MOIS = {1: "septembre", 2: "août", 3: "juillet", 4: "juin", 5: "mai", 6: "avril"}
PAY = [f"PAY_{n}" for n in range(1, 7)]
avant_corrections = original[PAY].copy()
journal = {}

def saine_anterieure(ligne, n):
    for k in range(n + 1, 7):
        if ligne[f"PAY_{k}"] <= 0:
            return ligne[f"PAY_{k}"]
    return None

# 10, étape 0 : PAY_n = 1 et solde du mois suivant nul ou créditeur -> codification saine antérieure (d'avril à août)
nb = 0
for n in range(6, 1, -1):
    for k in original.index[(original[f"PAY_{n}"] == 1) & (original[f"BILL_AMT{n - 1}"] <= 0)]:
        v = saine_anterieure(original.loc[k], n)
        if v is not None:
            original.at[k, f"PAY_{n}"] = v; nb += 1
journal["10 : solde du mois suivant nul (codification saine antérieure)"] = nb

# 10, étape 1 : encours nul (pas de facture du mois, mois précédent sain) -> codification du mois précédent (de mai à août)
nb = 0
for n in range(5, 1, -1):
    regle = (original[f"PAY_{n}"] == 1) & (original[f"BILL_AMT{n}"] <= 0) & (original[f"PAY_{n + 1}"] <= 0)
    original.loc[regle, f"PAY_{n}"] = original.loc[regle, f"PAY_{n + 1}"]; nb += int(regle.sum())
journal["10 : encours nul"] = nb

# 10, étape 3 : décisions de l'utilisateur (codification anticipée) ; l'étape 2 (impayés, règle du 09) n'est pas appliquée
DECISIONS = {6783: {4: 0, 3: 0, 2: 0}, 11498: {4: 2, 3: 2, 2: 2}, 892: {3: 0, 2: 0}, 1967: {3: 0, 2: 0},
             5645: {2: 0}, 19195: {2: 2}, 25176: {2: 2}}
nb = 0
for id_client, valeurs in DECISIONS.items():
    ligne = original.index[original["ID"] == id_client]
    assert len(ligne) == 1, id_client
    for n, v in valeurs.items():
        original.loc[ligne, f"PAY_{n}"] = v; nb += 1
journal["10 : décisions de l'utilisateur"] = nb
assert int((original[PAY[1:]] == 1).sum().sum()) == 0, "il reste des codifications 1 d'avril à août"
assert (original["PAY_1"] == avant_corrections["PAY_1"]).all(), "PAY_1 ne doit pas être corrigé"

display(pd.DataFrame({"Codifications modifiées": journal}))
print("Codifications modifiées par mois (avril à août) :",
      {MOIS[n]: int((original[f"PAY_{n}"] != avant_corrections[f"PAY_{n}"]).sum()) for n in range(6, 1, -1)})
print(f"Clients concernés : {int((original[PAY] != avant_corrections).any(axis=1).sum())}")

,Codifications modifiées
10 : solde du mois suivant nul (codification saine antérieure),0
10 : encours nul,21
10 : décisions de l'utilisateur,13


Codifications modifiées par mois (avril à août) : {'avril': 0, 'mai': 0, 'juin': 2, 'juillet': 4, 'août': 28}
Clients concernés : 28


## 1 ter. Construction de `PAY_1_corrigee`

Les modèles du `11c` (MLP et CatBoost) sont rechargés avec leurs seuils, puis appliqués aux clients codifiés 1 en septembre et en retard en août (codification d'août corrigée à l'étape 1 bis), avec les variables qu'ils ont apprises (colonnes relatives : `codif_m1` = août, `solde_m0` = solde de septembre, …).

In [3]:
import json, pickle

dossier_11c = Path.cwd() / "modeles" / "11c"
synthese_11c = json.loads((dossier_11c / "11c_synthese.json").read_text(encoding="utf-8"))
modeles_11c, seuils_11c = {}, {}
for nom, info in synthese_11c["modeles"].items():
    with open(dossier_11c / info["fichier"], "rb") as f:
        modeles_11c[nom] = pickle.load(f)
    seuils_11c[nom] = info["seuil"]
print("Modèles du 11c :", {nom: round(s, 3) for nom, s in seuils_11c.items()})

uns = original["PAY_1"] == 1
sains_aout = uns & (original["PAY_2"] <= 0)
retard_aout = uns & (original["PAY_2"] >= 2)
assert int(uns.sum()) == int(sains_aout.sum() + retard_aout.sum()), "un PAY_1 = 1 n'est ni sain ni en retard en août"

# Variables du 11c pour septembre : codifications regroupées (2 et plus -> 2), soldes et paiements, plafond
d = original.loc[retard_aout]
X_11c = pd.DataFrame(index=d.index)
for k in range(1, 6):
    X_11c[f"codif_m{k}"] = d[f"PAY_{1 + k}"].where(d[f"PAY_{1 + k}"] < 2, 2).astype(float)
for k in range(0, 6):
    X_11c[f"solde_m{k}"] = d[f"BILL_AMT{1 + k}"]
    X_11c[f"paiement_m{k}"] = d[f"PAY_AMT{1 + k}"]
X_11c["LIMIT_BAL"] = d["LIMIT_BAL"]
X_11c = X_11c[synthese_11c["variables"]]

annonce = {nom: modeles_11c[nom].predict_proba(X_11c)[:, 1] >= seuils_11c[nom] for nom in modeles_11c}
sortie = annonce["MLP"] & annonce["CatBoost"]

original["PAY_1_corrigee"] = original["PAY_1"]
original.loc[sains_aout, "PAY_1_corrigee"] = original.loc[sains_aout, "PAY_2"]
original.loc[retard_aout, "PAY_1_corrigee"] = np.where(sortie, -1, 2)
assert not (original["PAY_1_corrigee"] == 1).any()

statut = pd.Series("PAY_1 tranchée par la banque", index=original.index)
statut[sains_aout] = "1, sain en août : codification d'août"
statut[retard_aout] = np.where(sortie, "1, en retard en août : sortie (deux modèles)",
                               np.where(annonce["MLP"] | annonce["CatBoost"], "1, en retard en août : retard maintenu (un seul modèle annonce la sortie)",
                                        "1, en retard en août : retard maintenu (aucun modèle)"))
display(pd.crosstab(statut.rename("Statut"), original["PAY_1_corrigee"].rename("PAY_1_corrigee"), margins=True, margins_name="Total"))

dossier_sortie = Path.cwd() / "sorties"
dossier_sortie.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"ID": original["ID"], "PAY_1": original["PAY_1"], "PAY_1_corrigee": original["PAY_1_corrigee"], "statut": statut}).to_csv(
    dossier_sortie / "pay1_corrigee.csv", index=False)
print(f"Colonne enregistrée : {dossier_sortie / 'pay1_corrigee.csv'}")

Modèles du 11c : {'MLP': 0.63, 'CatBoost': 0.724}


PAY_1_corrigee,-2,-1,0,2,3,4,5,6,7,8,Total
Statut,,,,,,,,,,,
"1, en retard en août : retard maintenu (aucun modèle)",0,0,0,1635,0,0,0,0,0,0,1635
"1, en retard en août : retard maintenu (un seul modèle annonce la sortie)",0,0,0,60,0,0,0,0,0,0,60
"1, en retard en août : sortie (deux modèles)",0,132,0,0,0,0,0,0,0,0,132
"1, sain en août : codification d'août",1233,621,7,0,0,0,0,0,0,0,1861
PAY_1 tranchée par la banque,2759,5686,14737,2667,322,76,26,11,9,19,26312
Total,3992,6439,14744,4362,322,76,26,11,9,19,30000


Colonne enregistrée : c:\Users\johan\VS Code Wild 2\PROJET_DEFAULT_CREDIT_CARD_ML\lab_ML\correction_codification_1\sorties\pay1_corrigee.csv


**Comment lire** : une ligne par statut, une colonne par valeur de `PAY_1_corrigee`. Les clients dont la banque a tranché `PAY_1` gardent leur codification ; les 1 sont répartis selon la règle. Le statut n'entre pas dans les modèles : il sert seulement à lire la colonne.

## 1 quater. Variables des deux variantes

In [4]:
# ==============================================================================
# VARIABLES : les 23 variables d'origine, rangées selon leur encodage (celui des scénarios, méthode v8)
# ==============================================================================
PAY = [f"PAY_{n}" for n in range(1, 7)]

# PAY_n : nombres ordonnés (-2 à 8), mis à l'échelle (v6)
# PAY_AMTn : paiements très étirés, logarithme (v7) ; LIMIT_BAL aussi au logarithme dans le 11e
# BILL_AMTn : factures, logarithme signé, car elles peuvent être négatives (v8)
# SEX, EDUCATION, MARRIAGE : catégories sans ordre, une colonne par catégorie (regroupées au niveau 0)
TYPES_ORIGINE = {
    "num": ["AGE"] + PAY + ["PAY_1_corrigee"],   # PAY_1_corrigee encodée comme les autres PAY_n
    "log": [f"PAY_AMT{n}" for n in range(1, 7)] + ["LIMIT_BAL"],   # 11e : plafond au logarithme (décision de l'utilisateur)
    "log_signe": [f"BILL_AMT{n}" for n in range(1, 7)],
    "nom": ["SEX", "EDUCATION", "MARRIAGE"],
}

# Les trois variantes : colonnes gardées, chacune avec son encodage d'origine
# Deux variantes : les 23 variables ou les PAY_n seuls, chacune avec PAY_1_corrigee à côté de PAY_1
VARIANTES = {
    "Complet + PAY_1 corrigée": lambda c: True,
    "PAY_n seuls + PAY_1 corrigée": lambda c: c in PAY + ["PAY_1_corrigee"],
}
TYPES_VARIANTE = {v: {t: [c for c in cols if garder(c)] for t, cols in TYPES_ORIGINE.items()} for v, garder in VARIANTES.items()}
COLONNES_VARIANTE = {v: [c for cols in types.values() for c in cols] for v, types in TYPES_VARIANTE.items()}

# Contrôles : colonnes présentes, ni identifiant ni cible, les 23 variables couvertes par la variante complète
assert sorted(COLONNES_VARIANTE["Complet + PAY_1 corrigée"]) == sorted(c for c in original.columns if c not in ("ID", "dpnm"))
assert (original[TYPES_ORIGINE["log"]] >= 0).all().all(), "Le logarithme ne s'applique qu'à des montants positifs ou nuls"

display(pd.DataFrame({
    "Variante": list(VARIANTES),
    "Nombre de variables": [len(COLONNES_VARIANTE[v]) for v in VARIANTES],
    "Variables": [", ".join(COLONNES_VARIANTE[v]) for v in VARIANTES],
}).set_index("Variante"))

,Nombre de variables,Variables
Variante,,
Complet + PAY_1 corrigée,24,"AGE, PAY_1, PAY_2, PAY_3, PAY_4, PAY_5, PAY_6,..."
PAY_n seuls + PAY_1 corrigée,7,"PAY_1, PAY_2, PAY_3, PAY_4, PAY_5, PAY_6, PAY_..."


## 2. Pipeline, critère et boucle : repris des scénarios

Le pipeline et la boucle sont ceux des scénarios (`ml_13`, méthode v8), à quatre différences près : le **critère de la boucle**, la précision moyenne (`CRITERE`, au lieu de la précision au rappel minimal) ; le **seuil de surapprentissage à 7 %**, mesuré sur ce critère ; les **4 modèles** suivis ; un pipeline construit **pour chaque variante** (seules ses colonnes y entrent). Le plancher contre le sous-apprentissage ne change pas de sens : la précision moyenne d'un tri au hasard vaut le taux de défaut. Aucune valeur n'est remplacée : le jeu de base n'a pas de vide.

In [5]:
from catboost import CatBoostClassifier
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", None)   # textes affichés en entier

# Pondération des défauts pour CatBoost (part des non-défauts / part des défauts)
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

# Transformation de chaque type de colonne (un nouvel objet à chaque appel)
TRANSFORMATIONS = {
    "num": lambda: StandardScaler(),
    "log": lambda: make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()),
    "log_signe": lambda: make_pipeline(FunctionTransformer(np.arcsinh, feature_names_out="one-to-one"), StandardScaler()),
    "nom": lambda: OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
}

def preprocesseur(variante):
    # Une branche par type de colonne, seulement pour les types présents dans la variante
    return ColumnTransformer(transformers=[(nom, TRANSFORMATIONS[nom](), colonnes)
                                           for nom, colonnes in TYPES_VARIANTE[variante].items() if colonnes])

# Critère optimisé par la boucle : la précision moyenne (aire sous la courbe précision-rappel), sur tous les rappels
CRITERE = "average_precision"
scorers = {
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
}

# Rappel minimal des scénarios (60 %) : sert seulement à la comparaison avec les scénarios (section 6)
RAPPEL_MINIMAL = 0.60

def seuil_pour_rappel(y, proba, rappel=RAPPEL_MINIMAL):
    # Seuil le plus haut qui détecte au moins la part voulue des défauts
    proba_defauts = np.sort(proba[np.asarray(y) == 1])[::-1]
    return proba_defauts[int(np.ceil(rappel * len(proba_defauts))) - 1]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Modèles de l'essai : les meilleurs des scénarios, avec leurs réglages de la première itération
MODELES_A_TESTER = ["LogisticRegression", "MLPClassifier", "RandomForest", "CatBoost"]

def modeles_de_base(variante):
    prep = preprocesseur(variante)
    tous = {
        "LogisticRegression": make_pipeline(prep, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        "MLPClassifier": make_pipeline(prep, MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=500, random_state=42, early_stopping=True)),
        "RandomForest": make_pipeline(prep, RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)),
        "CatBoost": make_pipeline(prep, CatBoostClassifier(scale_pos_weight=scale_pos, random_state=42, verbose=0, thread_count=-1, allow_writing_files=False)),
    }
    return {nom: tous[nom] for nom in MODELES_A_TESTER}

# ------------------------------------------------------------------------------
# Alertes et tableaux de scores (ceux des scénarios)
SEUIL_SURAPPRENTISSAGE = 0.07   # écart relatif train - val du critère : 7 % pour cet essai (5 % dans les scénarios) ; le double : surapprentissage fort
SEUIL_INSTABILITE = 0.02        # écart-type du critère sur les plis au-delà duquel on alerte (alerte seule, pas une contrainte)
METRIQUES = {"average_precision": "Précision moyenne", "roc_auc": "ROC AUC"}
LIBELLE_SCORE = "Précision des défauts prédits"
TAUX_DEFAUT = y_train.mean()    # précision, et précision moyenne, d'un tri au hasard

def apprend(score_val, std_val):
    # Plancher contre le sous-apprentissage : le critère doit dépasser le hasard (taux de défaut) de plus d'un écart-type
    return score_val - TAUX_DEFAUT > std_val

def alerte(ecart):
    if ecart > 2 * SEUIL_SURAPPRENTISSAGE:
        return "⚠️⚠️ SURAPPRENTISSAGE FORT"
    if ecart > SEUIL_SURAPPRENTISSAGE:
        return "⚠️ SURAPPRENTISSAGE"
    return "✅ OK"

def ligne_scores(nom, moy_val, std_val, moy_train):
    # Une ligne de tableau : chaque métrique en entraînement et en validation, le critère, son écart-type et l'alerte
    ligne = {"Modèle": nom}
    for cle, libelle in METRIQUES.items():
        ligne[f"{libelle} (train)"] = moy_train[cle]
        ligne[f"{libelle} (val)"] = moy_val[cle]
    ligne["Critère (val)"], ligne["Critère (train)"] = moy_val[CRITERE], moy_train[CRITERE]
    ligne["Écart-type critère (val)"] = std_val
    ligne["Écart train - val"] = moy_train[CRITERE] - moy_val[CRITERE]
    ligne["Écart relatif"] = ligne["Écart train - val"] / moy_val[CRITERE]
    ligne["Apprend"] = apprend(moy_val[CRITERE], std_val)
    ligne["Alerte"] = alerte(ligne["Écart relatif"]) if ligne["Apprend"] else "⚠️ N'APPREND RIEN (niveau du hasard)"
    if std_val > SEUIL_INSTABILITE:
        ligne["Alerte"] += " ; ⚠️ INSTABLE"
    return ligne

def pct(serie):
    return (serie * 100).round(1)

def parametres(params):
    # Paramètres sur une ligne : « depth 4, 5, 6 ; l2_leaf_reg 15 »
    return " ; ".join(f"{p} {', '.join(map(str, v)) if isinstance(v, list) else v}" for p, v in params.items())

def montrer(df, titre):
    display(Markdown(f"**{titre}**"))
    display(df.set_index(df.columns[0]))

In [6]:
from itertools import product
from sklearn.model_selection import GridSearchCV

TOURS_SECURITE = 15   # garde-fou contre une boucle sans fin : la boucle s'arrête d'elle-même avant ; s'il est atteint, revoir les grilles ou les bornes

# Grilles du premier tour : celles des scénarios
grilles = {
    "LogisticRegression": {"logisticregression__C": [0.01, 0.1, 1.0, 10.0]},
    "MLPClassifier": {"mlpclassifier__hidden_layer_sizes": [(32, 16), (64, 32), (32,)], "mlpclassifier__alpha": [0.01, 0.1, 1.0]},
    "RandomForest": {"randomforestclassifier__max_depth": [6, 8, 10], "randomforestclassifier__min_samples_leaf": [20, 50, 100], "randomforestclassifier__n_estimators": [200, 300]},
    "CatBoost": {"catboostclassifier__depth": [4, 6], "catboostclassifier__l2_leaf_reg": [5, 10, 15], "catboostclassifier__learning_rate": [0.03, 0.05],
                 "catboostclassifier__iterations": [1000]},   # 1 000 arbres : valeur par défaut, réduite par la boucle en cas de surapprentissage
}

# Comment faire évoluer chaque paramètre numérique d'un tour à l'autre (ceux des scénarios) :
# - simplifier : sens qui réduit le surapprentissage (+1 = augmenter, -1 = diminuer) ;
# - pas : écart entre deux valeurs (« x » = multiplier, « + » = ajouter) ;
# - bornes : valeurs minimale et maximale autorisées.
# Les paramètres absents de ce tableau (architecture du MLP, nombre d'arbres du RandomForest) sont fixés à leur meilleure valeur après le premier tour.
REGLAGES_PARAMETRES = {
    "logisticregression__C":                    {"simplifier": -1, "pas": ("x", 3),   "bornes": (1e-4, 1e4),  "entier": False},
    "mlpclassifier__alpha":                     {"simplifier": +1, "pas": ("x", 3),   "bornes": (1e-5, 100),  "entier": False},
    "randomforestclassifier__max_depth":        {"simplifier": -1, "pas": ("+", 2),   "bornes": (2, 30),      "entier": True},
    "randomforestclassifier__min_samples_leaf": {"simplifier": +1, "pas": ("x", 1.5), "bornes": (1, 2000),    "entier": True},
    "catboostclassifier__depth":                {"simplifier": -1, "pas": ("+", 1),   "bornes": (2, 10),      "entier": True},
    "catboostclassifier__l2_leaf_reg":          {"simplifier": +1, "pas": ("x", 1.5), "bornes": (1, 200),     "entier": False},
    "catboostclassifier__learning_rate":        {"simplifier": -1, "pas": ("x", 1.5), "bornes": (0.003, 0.3), "entier": False},
    "catboostclassifier__iterations":           {"simplifier": -1, "pas": ("x", 1.5), "bornes": (100, 3000),  "entier": True},
}

def nom_court(parametre):
    return parametre.split("__")[-1]

def arrondir(valeur, entier):
    return int(round(valeur)) if entier else float(f"{valeur:.3g}")

def voisin(valeur, reglage, sens):
    # Valeur suivante dans un sens (+1 = plus grande, -1 = plus petite), dans les bornes
    mode, pas = reglage["pas"]
    if mode == "x":
        nouvelle = valeur * pas if sens > 0 else valeur / pas
    else:
        nouvelle = valeur + pas if sens > 0 else valeur - pas
    bas, haut = reglage["bornes"]
    nouvelle = arrondir(min(max(nouvelle, bas), haut), reglage["entier"])
    return None if nouvelle == valeur else nouvelle

def choisir_index(r):
    # Choix sous contrainte, parmi les réglages qui battent le hasard : le meilleur critère sans surapprentissage ;
    # sinon le moins surappris ; si aucun ne bat le hasard, le meilleur score
    val, std = r[f"mean_test_{CRITERE}"], r[f"std_test_{CRITERE}"]
    ecarts = (r[f"mean_train_{CRITERE}"] - val) / val   # écarts relatifs
    utiles = apprend(val, std)
    sous_seuil = np.where(utiles & (ecarts <= SEUIL_SURAPPRENTISSAGE))[0]
    if len(sous_seuil):
        return int(sous_seuil[np.argmax(val[sous_seuil])])
    if utiles.any():
        idx = np.where(utiles)[0]
        return int(idx[np.argmin(ecarts[idx])])
    return int(np.argmax(val))

def bords_de_grille(grille, meilleurs):
    # Paramètres numériques dont la meilleure valeur est la plus petite ou la plus grande de la grille (+1 = maximum, -1 = minimum)
    bords = {}
    for p, valeurs in grille.items():
        if p in REGLAGES_PARAMETRES and len(valeurs) > 1:
            if meilleurs[p] == max(valeurs):
                bords[p] = +1
            elif meilleurs[p] == min(valeurs):
                bords[p] = -1
    return bords

def grille_suivante(grille, meilleurs, bords, surapprend):
    # Grille du tour suivant et explication des corrections ; None si plus rien ne peut bouger
    nouvelle, actions = {}, []
    for p, valeurs in grille.items():
        v = meilleurs[p]
        reglage = REGLAGES_PARAMETRES.get(p)
        if reglage is None:
            nouvelle[p] = [v]                            # catégorie ou paramètre sans effet : fixé
            continue
        sens = None
        if surapprend and reglage["simplifier"] != 0:
            sens = reglage["simplifier"]                 # pousser vers un modèle plus simple
            raison = "surapprentissage"
        elif p in bords:
            sens = bords[p]                              # prolonger la grille du côté du bord
            raison = "bord de grille"
        if sens is None:
            nouvelle[p] = [v]
            continue
        v1 = voisin(v, reglage, sens)
        v2 = voisin(v1, reglage, sens) if v1 is not None else None
        valeurs_tour = [x for x in (v, v1, v2) if x is not None]
        nouvelle[p] = sorted(set(valeurs_tour))
        if len(nouvelle[p]) > 1:
            actions.append(f"{nom_court(p)} {v} → {nouvelle[p]} ({raison})")
    if not actions:
        return None, ["bornes atteintes : la grille ne peut plus bouger"]
    return nouvelle, actions

def lancer_grille(nom, modele, grille, X):
    grid = GridSearchCV(modele, grille, cv=cv, scoring=scorers, refit=choisir_index, return_train_score=True, n_jobs=-1)
    grid.fit(X, y_train)
    r, i = grid.cv_results_, grid.best_index_
    ligne = ligne_scores(nom,
                         {k: r[f"mean_test_{k}"][i] for k in METRIQUES},
                         r[f"std_test_{CRITERE}"][i],
                         {k: r[f"mean_train_{k}"][i] for k in METRIQUES})
    ligne["Meilleurs paramètres"] = {nom_court(k): v for k, v in grid.best_params_.items()}
    return grid, ligne

def cle_reglage(params):
    # Identifiant d'une combinaison de paramètres, pour savoir si elle a déjà été testée
    return str(sorted(params.items()))

def reglages_nouveaux(grille, deja_testes):
    # Combinaisons de la grille qui n'ont encore été testées à aucun tour
    combinaisons = [dict(zip(grille, valeurs)) for valeurs in product(*grille.values())]
    return [c for c in combinaisons if cle_reglage(c) not in deja_testes]

def choisir_tour(tours):
    # Version retenue parmi les tours joués : d'abord ceux qui battent le hasard et sans surapprentissage
    # (sinon ceux qui battent le hasard, sinon tous) ; parmi eux, ceux à moins d'un écart-type du meilleur score
    # (à égalité avec lui) ; et parmi ces derniers, celui qui surapprend le moins
    for garder in (lambda l: l["Apprend"] and l["Écart relatif"] <= SEUIL_SURAPPRENTISSAGE, lambda l: l["Apprend"], lambda l: True):
        groupe = [t for t in tours if garder(t[1])]
        if groupe:
            break
    meilleur = max(groupe, key=lambda t: t[1]["Critère (val)"])[1]
    egaux = [t for t in groupe if meilleur["Critère (val)"] - t[1]["Critère (val)"] <= meilleur["Écart-type critère (val)"]]
    return min(egaux, key=lambda t: t[1]["Écart relatif"])

def boucle(variante):
    # Boucle des scénarios, pour un jeu de variables : version retenue de chaque modèle, lignes de scores et historique
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    best_models, lignes_retenues, historique = {}, [], []
    for nom, modele in modeles_de_base(variante).items():
        grille, retenu, fin, deja_testes, reglage_precedent, tours = grilles[nom], None, None, set(), None, []
        for tour in range(1, TOURS_SECURITE + 1):
            grid, ligne = lancer_grille(nom, modele, grille, X)
            ligne["Tour"] = tour
            deja_testes |= {cle_reglage(c) for c in grid.cv_results_["params"]}
            surapprend = ligne["Écart relatif"] > SEUIL_SURAPPRENTISSAGE
            bords = bords_de_grille(grille, grid.best_params_)
            precedent = retenu
            tours.append((grid, ligne))
            retenu = choisir_tour(tours)
            grille_tour = {nom_court(p): v for p, v in grille.items()}
            reglage = ligne["Meilleurs paramètres"]
            # Stagnation : seulement entre deux tours sans surapprentissage
            stagne = (precedent is not None and not surapprend
                      and precedent[1]["Écart relatif"] <= SEUIL_SURAPPRENTISSAGE
                      and ligne["Critère (val)"] - precedent[1]["Critère (val)"] <= precedent[1]["Écart-type critère (val)"])
            if not ligne["Apprend"]:
                fin = "sous-apprentissage : le réglage choisi ne bat plus le hasard"
            elif not surapprend and not bords:
                fin = "optimisé : ni surapprentissage ni paramètre au bord"
            elif reglage == reglage_precedent:
                fin = "convergé : le réglage choisi ne change plus"
            elif stagne:
                fin = "stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage"
            elif tour == TOURS_SECURITE:
                fin = "⚠️ garde-fou atteint : grilles ou bornes à revoir"
            else:
                grille, actions = grille_suivante(grille, grid.best_params_, bords, surapprend)
                if grille is None:
                    fin = actions[0]
                elif not reglages_nouveaux(grille, deja_testes):
                    fin = "convergé : la grille suivante ne contient aucun réglage nouveau"
            reglage_precedent = reglage
            historique.append({
                "Modèle": nom, "Tour": tour, "Grille testée": grille_tour, "Réglage choisi": reglage,
                "Critère (val)": ligne["Critère (val)"], "Écart-type": ligne["Écart-type critère (val)"],
                "Écart relatif": ligne["Écart relatif"], "Alerte": ligne["Alerte"],
                "Suite": fin if fin else "; ".join(actions),
            })
            if fin:
                break
        best_models[nom] = retenu[0].best_estimator_
        lignes_retenues.append(dict(retenu[1], **{"Arrêt": fin, "Tours joués": tour}))
        print(f"  {nom} : {tour} tour(s), {fin} ; version retenue : tour {retenu[1]['Tour']}")
    return best_models, pd.DataFrame(lignes_retenues), pd.DataFrame(historique)

# Affichage : grille du premier tour, sens de simplification, pas et bornes de chaque paramètre
lignes = []
for modele, grille in grilles.items():
    for parametre, valeurs in grille.items():
        r = REGLAGES_PARAMETRES.get(parametre)
        lignes.append({"Modèle": modele, "Paramètre": nom_court(parametre), "Premier tour": ", ".join(map(str, valeurs)),
                       "Pour simplifier": ("augmenter" if r["simplifier"] > 0 else "diminuer") if r else "fixé après le premier tour",
                       "Pas": (f"× {r['pas'][1]}" if r["pas"][0] == "x" else f"+ {r['pas'][1]}") if r else "—",
                       "Borne min": r["bornes"][0] if r else "—", "Borne max": r["bornes"][1] if r else "—"})
display(Markdown("**Bornes et pas de chaque paramètre** (la boucle ne sort jamais de ces bornes)"))
display(pd.DataFrame(lignes).set_index(["Modèle", "Paramètre"]))


**Bornes et pas de chaque paramètre** (la boucle ne sort jamais de ces bornes)

Premier tour             Pour simplifier    Pas Borne min Borne max
Modèle             Paramètre                                                                                           
LogisticRegression C                        0.01, 0.1, 1.0, 10.0                    diminuer    × 3    0.0001   10000.0
MLPClassifier      hidden_layer_sizes  (32, 16), (64, 32), (32,)  fixé après le premier tour      —         —         —
                   alpha                          0.01, 0.1, 1.0                   augmenter    × 3   0.00001       100
RandomForest       max_depth                            6, 8, 10                    diminuer    + 2         2        30
                   min_samples_leaf                  20, 50, 100                   augmenter  × 1.5         1      2000
                   n_estimators                         200, 300  fixé après le premier tour      —         —         —
CatBoost           depth                                    4, 6                    diminuer    + 1         2        10
                   l2_leaf_reg                         5, 10, 15                   augmenter  × 1.5         1       200
                   learning_rate                      0.03, 0.05                    diminuer  × 1.5     0.003       0.3
                   iterations                               1000                    diminuer  × 1.5       100      3000

## 3. Boucle d'optimisation, variante par variante

Pour chaque variante, la boucle règle les 4 modèles l'un après l'autre. Une ligne s'affiche pendant le calcul pour chaque modèle (nombre de tours, raison de l'arrêt, tour retenu). Puis, pour chaque variante :
- **Version retenue de chaque modèle** : la précision moyenne en validation (le critère), son écart-type, l'entraînement face à la validation, l'écart relatif (au-delà de 7 % : ⚠️ ; de 14 % : ⚠️⚠️), le ROC AUC en contrôle, et le réglage retenu ;
- **Historique de la boucle** : une ligne par tour et par modèle, avec la grille testée, le réglage choisi et la correction qui suit (ou la raison de l'arrêt).

Le calcul est long : la boucle tourne deux fois, sur 30 000 clients.

In [7]:
import time

resultats = {}
for variante in VARIANTES:
    debut = time.time()
    print(f"Variante « {variante} » ({len(COLONNES_VARIANTE[variante])} variables)")
    resultats[variante] = boucle(variante)
    print(f"  durée : {(time.time() - debut) / 60:.1f} min")

for variante, (_, df_retenus, df_historique) in resultats.items():
    display(Markdown(f"### Variante « {variante} »"))
    df = df_retenus.sort_values("Critère (val)", ascending=False)
    montrer(pd.DataFrame({
        "Modèle": df["Modèle"],
        "Précision moyenne val (%)": pct(df["Critère (val)"]),
        "Écart-type (pts)": pct(df["Écart-type critère (val)"]),
        "Précision moyenne train (%)": pct(df["Critère (train)"]),
        "Écart relatif (%)": pct(df["Écart relatif"]),
        "ROC AUC val": df["ROC AUC (val)"].round(3),
        "ROC AUC train": df["ROC AUC (train)"].round(3),
        "Alerte": df["Alerte"],
        "Tours joués": df["Tours joués"],
        "Tour retenu": df["Tour"],
        "Raison de l'arrêt": df["Arrêt"],
        "Réglage retenu": df["Meilleurs paramètres"].map(parametres),
    }), "Version retenue de chaque modèle (validation)")
    montrer(pd.DataFrame({
        "Modèle": df_historique["Modèle"],
        "Tour": df_historique["Tour"],
        "Grille testée": df_historique["Grille testée"].map(parametres),
        "Réglage choisi": df_historique["Réglage choisi"].map(parametres),
        "Précision moyenne val (%)": pct(df_historique["Critère (val)"]),
        "Écart relatif (%)": pct(df_historique["Écart relatif"]),
        "Alerte": df_historique["Alerte"],
        "Suite": df_historique["Suite"],
    }), "Historique de la boucle")

Variante « Complet + PAY_1 corrigée » (24 variables)


  LogisticRegression : 2 tour(s), stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage ; version retenue : tour 2
  MLPClassifier : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1
  RandomForest : 2 tour(s), convergé : le réglage choisi ne change plus ; version retenue : tour 1
  CatBoost : 3 tour(s), convergé : le réglage choisi ne change plus ; version retenue : tour 2
  durée : 6.7 min
Variante « PAY_n seuls + PAY_1 corrigée » (7 variables)
  LogisticRegression : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1
  MLPClassifier : 2 tour(s), stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage ; version retenue : tour 1
  RandomForest : 2 tour(s), stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage ; version retenue : tour 1
  CatBoost : 2 tour(s), stagnation : gain plus petit que l'écart-type, entre deux

### Variante « Complet + PAY_1 corrigée »

**Version retenue de chaque modèle (validation)**

,Précision moyenne val (%),Écart-type (pts),Précision moyenne train (%),Écart relatif (%),ROC AUC val,ROC AUC train,Alerte,Tours joués,Tour retenu,Raison de l'arrêt,Réglage retenu
Modèle,,,,,,,,,,,
CatBoost,55.6,0.8,59.3,6.8,0.784,0.815,✅ OK,3,2,convergé : le réglage choisi ne change plus,depth 4 ; iterations 667 ; l2_leaf_reg 22.5 ; learning_rate 0.03
RandomForest,55.5,0.7,58.4,5.3,0.777,0.791,✅ OK,2,1,convergé : le réglage choisi ne change plus,max_depth 6 ; min_samples_leaf 20 ; n_estimators 300
MLPClassifier,53.9,0.8,55.3,2.4,0.775,0.785,✅ OK,1,1,optimisé : ni surapprentissage ni paramètre au bord,"alpha 0.1 ; hidden_layer_sizes (32, 16)"
LogisticRegression,50.4,0.5,50.5,0.2,0.753,0.755,✅ OK,2,2,"stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage",C 0.00111


...

### Variante « PAY_n seuls + PAY_1 corrigée »

**Version retenue de chaque modèle (validation)**

,Précision moyenne val (%),Écart-type (pts),Précision moyenne train (%),Écart relatif (%),ROC AUC val,ROC AUC train,Alerte,Tours joués,Tour retenu,Raison de l'arrêt,Réglage retenu
Modèle,,,,,,,,,,,
RandomForest,53.2,0.7,53.9,1.4,0.749,0.751,✅ OK,2,1,"stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage",max_depth 6 ; min_samples_leaf 20 ; n_estimators 300
CatBoost,53.1,0.6,54.1,2.0,0.748,0.755,✅ OK,2,2,"stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage",depth 4 ; iterations 1000 ; l2_leaf_reg 33.8 ; learning_rate 0.0133
MLPClassifier,52.1,0.7,52.5,0.9,0.743,0.746,✅ OK,2,1,"stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage","alpha 0.01 ; hidden_layer_sizes (64, 32)"
LogisticRegression,48.6,0.8,48.5,-0.1,0.704,0.705,✅ OK,1,1,optimisé : ni surapprentissage ni paramètre au bord,C 0.1


**Historique de la boucle**

,Tour,Grille testée,Réglage choisi,Précision moyenne val (%),Écart relatif (%),Alerte,Suite
Modèle,,,,,,,
LogisticRegression,1,"C 0.01, 0.1, 1.0, 10.0",C 0.1,48.6,-0.1,✅ OK,optimisé : ni surapprentissage ni paramètre au bord
MLPClassifier,1,"hidden_layer_sizes (32, 16), (64, 32), (32,) ; alpha 0.01, 0.1, 1.0","alpha 0.01 ; hidden_layer_sizes (64, 32)",52.1,0.9,✅ OK,"alpha 0.01 → [0.00111, 0.00333, 0.01] (bord de grille)"
MLPClassifier,2,"hidden_layer_sizes (64, 32) ; alpha 0.00111, 0.00333, 0.01","alpha 0.00111 ; hidden_layer_sizes (64, 32)",52.2,1.2,✅ OK,"stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage"
RandomForest,1,"max_depth 6, 8, 10 ; min_samples_leaf 20, 50, 100 ; n_estimators 200, 300",max_depth 6 ; min_samples_leaf 20 ; n_estimators 300,53.2,1.4,✅ OK,"max_depth 6 → [2, 4, 6] (bord de grille); min_samples_leaf 20 → [9, 13, 20] (bord de grille)"
RandomForest,2,"max_depth 2, 4, 6 ; min_samples_leaf 9, 13, 20 ; n_estimators 300",max_depth 6 ; min_samples_leaf 9 ; n_estimators 300,53.2,1.6,✅ OK,"stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage"
CatBoost,1,"depth 4, 6 ; l2_leaf_reg 5, 10, 15 ; learning_rate 0.03, 0.05 ; iterations 1000",depth 4 ; iterations 1000 ; l2_leaf_reg 15 ; learning_rate 0.03,53.0,3.8,✅ OK,"depth 4 → [2, 3, 4] (bord de grille); l2_leaf_reg 15 → [15, 22.5, 33.8] (bord de grille); learning_rate 0.03 → [0.0133, 0.02, 0.03] (bord de grille)"
CatBoost,2,"depth 2, 3, 4 ; l2_leaf_reg 15, 22.5, 33.8 ; learning_rate 0.0133, 0.02, 0.03 ; iterations 1000",depth 4 ; iterations 1000 ; l2_leaf_reg 33.8 ; learning_rate 0.0133,53.1,2.0,✅ OK,"stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage"


**Lecture** : une section par variante. Le taux de défaut du jeu de base est la précision moyenne d'un tri au hasard : une précision moyenne proche de lui veut dire que le modèle ne trouve presque rien. Un modèle qui reste en surapprentissage malgré tous ses tours est signalé (⚠️) : sa grille de départ ou ses bornes sont peut-être à revoir pour ces données.

## 4. Seuil de décision optimisé et comparaison des variantes, pli par pli

**Seuil optimisé automatiquement** : sur chaque pli, la version retenue de chaque modèle est réentraînée sur les 4 autres plis, enveloppée dans `TunedThresholdClassifierCV`. Celui-ci découpe à son tour ces 4 plis en 5 plis internes et essaie une série de seuils sur les prédictions internes. Il garde celui qui maximise le **gain net** « défauts détectés − `BONS_CLIENTS_PAR_DEFAUT` × bons clients signalés », puis réentraîne le modèle sur les 4 plis. Le pli de validation n'intervient ni dans les réglages du modèle ni dans le choix du seuil : il juge seulement le résultat. Chaque client reçoit une probabilité hors pli et une décision (signalé ou non) prise au seuil de son pli. Les modèles entraînés sur chaque pli sont gardés en mémoire pour la section 5.

**Contrôle de la règle** : la précision **marginale** se mesure sur les clients signalés les plus proches du seuil (les 10 % de clients signalés qui ont les probabilités les plus basses). Elle doit tourner autour de 1 / (1 + `BONS_CLIENTS_PAR_DEFAUT`), soit 50 % : c'est la part de défauts parmi les derniers clients que la règle a jugé bon d'ajouter.

Trois tableaux :
- **Détection au seuil optimisé** : pour chaque modèle et chaque variante, le seuil choisi (moyenne et étendue sur les 5 plis), le **rappel** (part des défauts détectés), la **précision des défauts prédits** (sur tous les clients signalés), la **précision marginale** (sur les 10 % de signalés les plus proches du seuil), la part des clients signalés et le F1 (pour information), en moyenne sur les plis avec leur écart-type ; puis la précision moyenne et le ROC AUC, qui ne dépendent d'aucun seuil ;
- **Comparaison avec la variante complète** : pour chaque modèle, puis en meilleur contre meilleur, le gain moyen sur la variante complète, son écart-type et les plis gagnés, pour le rappel au seuil optimisé (combien de défauts sont détectés avec la règle) et pour la précision moyenne (la capacité de détection, tous seuils confondus). Verdict des scénarios : « gain réel » si le gain moyen dépasse l'écart-type des gains, « perte » s'il est en dessous de moins cet écart-type, sinon « dans le bruit ».

In [8]:
from sklearn.base import clone
from sklearn.metrics import average_precision_score, fbeta_score, make_scorer, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import TunedThresholdClassifierCV

y = y_train.to_numpy()
plis = list(cv.split(original, y_train))   # les mêmes 5 plis que la boucle
cv_interne = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # plis internes, pour le choix du seuil

# Règle fixée à l'avance par le métier : au plus 1 bon client signalé pour 1 défaut attrapé, à la marge
BONS_CLIENTS_PAR_DEFAUT = 1
PRECISION_MARGINALE_VISEE = 1 / (1 + BONS_CLIENTS_PAR_DEFAUT)   # 50 % : précision des derniers clients ajoutés au seuil retenu
PART_MARGE = 0.10   # contrôle : précision mesurée sur les 10 % de clients signalés les plus proches du seuil

def gain_net(y_vrai, y_pred):
    # Défauts détectés moins les bons clients signalés (pondérés), rapporté au nombre de clients :
    # maximal au seuil où les clients ajoutés ne sont plus en défaut qu'avec la précision marginale visée
    y_vrai, y_pred = np.asarray(y_vrai), np.asarray(y_pred)
    detectes = (y_pred & (y_vrai == 1)).sum()
    fausses_alertes = (y_pred & (y_vrai == 0)).sum()
    return (detectes - BONS_CLIENTS_PAR_DEFAUT * fausses_alertes) / len(y_vrai)

SCORE_SEUIL = make_scorer(gain_net)

def precision_marginale(y_vrai, proba, signales):
    # Part de défauts parmi les PART_MARGE des clients signalés qui ont les probabilités les plus basses (les plus proches du seuil)
    idx = np.where(signales)[0]
    if len(idx) == 0:
        return np.nan
    derniers = idx[np.argsort(proba[idx])[:max(1, int(np.ceil(PART_MARGE * len(idx))))]]
    return y_vrai[derniers].mean()

def f2_au_rappel_minimal(y_vrai, proba):
    # Mesure des scénarios (section 6) : F2 au seuil qui détecte au moins le rappel minimal
    return fbeta_score(y_vrai, proba >= seuil_pour_rappel(y_vrai, proba), beta=2)

probas, mesures, modeles_plis = {}, {}, {}
for variante, (best_models, _, _) in resultats.items():
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    for nom, modele in best_models.items():
        p, entraines = np.zeros(len(y)), []
        m_plis = {"Seuil": [], "Rappel": [], "Précision": [], "Précision marginale": [], "Signalés": [], "F1": [], "Précision moyenne": [], "ROC AUC": []}
        for entrainement, validation in plis:
            seuil_auto = TunedThresholdClassifierCV(clone(modele), scoring=SCORE_SEUIL, cv=cv_interne, n_jobs=-1)
            seuil_auto.fit(X.iloc[entrainement], y[entrainement])
            p[validation] = seuil_auto.predict_proba(X.iloc[validation])[:, 1]
            decision = seuil_auto.predict(X.iloc[validation]).astype(bool)
            y_val = y[validation]
            m_plis["Seuil"].append(seuil_auto.best_threshold_)
            m_plis["Rappel"].append(recall_score(y_val, decision))
            m_plis["Précision"].append(precision_score(y_val, decision, zero_division=0))
            m_plis["Précision marginale"].append(precision_marginale(y_val, p[validation], decision))
            m_plis["Signalés"].append(decision.mean())
            m_plis["F1"].append(fbeta_score(y_val, decision, beta=1))
            m_plis["Précision moyenne"].append(average_precision_score(y_val, p[validation]))
            m_plis["ROC AUC"].append(roc_auc_score(y_val, p[validation]))
            entraines.append(seuil_auto)
        probas[(variante, nom)], mesures[(variante, nom)], modeles_plis[(variante, nom)] = p, m_plis, entraines
        print(f"{variante}, {nom} : seuil {np.mean(m_plis['Seuil']):.3f}, rappel {np.mean(m_plis['Rappel']) * 100:.1f} %, "
              f"précision {np.mean(m_plis['Précision']) * 100:.1f} %, précision marginale {np.nanmean(m_plis['Précision marginale']) * 100:.1f} %")

def moy_et(valeurs, en_pct=False):
    facteur = 100 if en_pct else 1
    return f"{np.mean(valeurs) * facteur:.{1 if en_pct else 3}f} ± {np.std(valeurs) * facteur:.{1 if en_pct else 3}f}"

lignes = []
for (variante, nom), m in mesures.items():
    lignes.append({
        "Variante": variante, "Modèle": nom,
        "Seuil (moyenne)": round(np.mean(m["Seuil"]), 3),
        "Seuil (min - max)": f"{min(m['Seuil']):.3f} - {max(m['Seuil']):.3f}",
        "Rappel (%)": moy_et(m["Rappel"], True),
        f"{LIBELLE_SCORE} (%)": moy_et(m["Précision"], True),
        f"Précision marginale, {PART_MARGE:.0%} les plus proches du seuil (%)": moy_et(m["Précision marginale"], True),
        "Clients signalés (%)": moy_et(m["Signalés"], True),
        "F1": moy_et(m["F1"]),
        "Précision moyenne": moy_et(m["Précision moyenne"]),
        "ROC AUC": moy_et(m["ROC AUC"]),
    })
display(Markdown(f"**Détection au seuil optimisé (au plus {BONS_CLIENTS_PAR_DEFAUT} bon(s) client(s) pour 1 défaut à la marge, "
                 f"précision marginale visée {PRECISION_MARGINALE_VISEE:.0%}), en validation** (moyenne ± écart-type sur les 5 plis ; taux de défaut : {TAUX_DEFAUT * 100:.1f} %)"))
display(pd.DataFrame(lignes).set_index(["Variante", "Modèle"]))

Complet + PAY_1 corrigée, LogisticRegression : seuil 0.678, rappel 38.8 %, précision 60.6 %, précision marginale 52.3 %
Complet + PAY_1 corrigée, MLPClassifier : seuil 0.509, rappel 35.8 %, précision 66.3 %, précision marginale 49.6 %
Complet + PAY_1 corrigée, RandomForest : seuil 0.773, rappel 36.2 %, précision 67.4 %, précision marginale 50.7 %
Complet + PAY_1 corrigée, CatBoost : seuil 0.779, rappel 37.0 %, précision 66.5 %, précision marginale 53.4 %
PAY_n seuls + PAY_1 corrigée, LogisticRegression : seuil 0.688, rappel 36.4 %, précision 67.0 %, précision marginale 51.6 %
PAY_n seuls + PAY_1 corrigée, MLPClassifier : seuil 0.520, rappel 35.2 %, précision 68.0 %, précision marginale 53.8 %
PAY_n seuls + PAY_1 corrigée, RandomForest : seuil 0.788, rappel 35.9 %, précision 68.6 %, précision marginale 61.3 %
PAY_n seuls + PAY_1 corrigée, CatBoost : seuil 0.792, rappel 35.7 %, précision 68.1 %, précision marginale 53.2 %


**Détection au seuil optimisé (au plus 1 bon(s) client(s) pour 1 défaut à la marge, précision marginale visée 50%), en validation** (moyenne ± écart-type sur les 5 plis ; taux de défaut : 22.1 %)

Seuil (moyenne) Seuil (min - max)  Rappel (%) Précision des défauts prédits (%) Précision marginale, 10% les plus proches du seuil (%) Clients signalés (%)             F1 Précision moyenne  \
Variante                     Modèle                                                                                                                                                                                                             
Complet + PAY_1 corrigée     LogisticRegression            0.678     0.673 - 0.684  38.8 ± 0.9                        60.6 ± 0.9                                             52.3 ± 3.9           14.2 ± 0.5  0.473 ± 0.004     0.504 ± 0.005   
                             MLPClassifier                 0.509     0.486 - 0.549  35.8 ± 2.0                        66.3 ± 2.5                                             49.6 ± 5.2           12.0 ± 0.8  0.465 ± 0.020     0.539 ± 0.008   
                             RandomForest                  0.773     0.756 - 0.783  36.2 ± 0.9                        67.4 ± 2.2                                             50.7 ± 9.7           11.9 ± 0.4  0.471 ± 0.010     0.555 ± 0.007   
                             CatBoost                      0.779     0.749 - 0.801  37.0 ± 2.2                        66.5 ± 0.8                                             53.4 ± 5.3           12.3 ± 0.8  0.475 ± 0.017     0.556 ± 0.008   
PAY_n seuls + PAY_1 corrigée LogisticRegression            0.688     0.679 - 0.701  36.4 ± 1.5                        67.0 ± 2.5                                             51.6 ± 7.9           12.1 ± 0.9  0.471 ± 0.009     0.486 ± 0.008   
                             MLPClassifier                 0.520     0.491 - 0.537  35.2 ± 1.5                        68.0 ± 1.1                                             53.8 ± 5.3           11.5 ± 0.4  0.464 ± 0.015     0.521 ± 0.007   
                             RandomForest                  0.788     0.783 - 0.791  35.9 ± 1.1                        68.6 ± 1.7                                             61.3 ± 4.0           11.6 ± 0.1  0.472 ± 0.013     0.532 ± 0.007   
                             CatBoost                      0.792     0.787 - 0.799  35.7 ± 1.1                        68.1 ± 1.9                                             53.2 ± 2.4           11.6 ± 0.2  0.468 ± 0.013     0.531 ± 0.006   

                                                       ROC AUC  
Variante                     Modèle                             
Complet + PAY_1 corrigée     LogisticRegression  0.753 ± 0.005  
                             MLPClassifier       0.775 ± 0.005  
                             RandomForest        0.777 ± 0.006  
                             CatBoost            0.784 ± 0.005  
PAY_n seuls + PAY_1 corrigée LogisticRegression  0.704 ± 0.005  
                             MLPClassifier       0.743 ± 0.008  
                             RandomForest        0.749 ± 0.006  
                             CatBoost            0.748 ± 0.007

In [9]:
def verdict(gains):
    moyenne, ecart_type = np.mean(gains), np.std(gains)
    return "✅ gain réel" if moyenne > ecart_type else ("❌ perte" if moyenne < -ecart_type else "➖ dans le bruit")

REFERENCE = "Complet + PAY_1 corrigée"
for mesure in ("Rappel", "Précision moyenne"):
    libelle = "Rappel au seuil optimisé" if mesure == "Rappel" else mesure
    def meilleur(variante):
        return max(MODELES_A_TESTER, key=lambda nom: np.mean(mesures[(variante, nom)][mesure]))
    lignes = []
    for variante in [v for v in VARIANTES if v != REFERENCE]:
        for nom in MODELES_A_TESTER + ["Meilleur contre meilleur"]:
            ref, comp = (meilleur(REFERENCE), meilleur(variante)) if nom == "Meilleur contre meilleur" else (nom, nom)
            score_ref, score_comp = np.array(mesures[(REFERENCE, ref)][mesure]), np.array(mesures[(variante, comp)][mesure])
            gains = score_comp - score_ref
            lignes.append({
                "Variante": variante,
                "Modèle": nom if nom != "Meilleur contre meilleur" else f"{nom} ({ref} → {comp})",
                f"{libelle}, complet": round(score_ref.mean(), 3),
                f"{libelle}, variante": round(score_comp.mean(), 3),
                "Gain": f"{gains.mean():+.3f} ± {gains.std():.3f}",
                "Part du score complet gardée (%)": round(score_comp.mean() / score_ref.mean() * 100, 1),
                "Plis gagnés": f"{(gains > 0).sum()} sur 5",
                "Verdict": verdict(gains),
            })
    display(Markdown(f"**Comparaison avec la variante complète, pli par pli : {libelle}**"))
    display(pd.DataFrame(lignes).set_index(["Variante", "Modèle"]))

**Comparaison avec la variante complète, pli par pli : Rappel au seuil optimisé**

Rappel au seuil optimisé, complet  Rappel au seuil optimisé, variante            Gain  Part du score complet gardée (%) Plis gagnés          Verdict
Variante                     Modèle                                                                                                                                                                                                                  
PAY_n seuls + PAY_1 corrigée LogisticRegression                                                                              0.388                               0.364  -0.023 ± 0.020                              94.0     1 sur 5          ❌ perte
                             MLPClassifier                                                                                   0.358                               0.352  -0.006 ± 0.021                              98.3     2 sur 5  ➖ dans le bruit
                             RandomForest                                                                                    0.362                               0.359  -0.003 ± 0.009                              99.2     2 sur 5  ➖ dans le bruit
                             CatBoost                                                                                        0.370                               0.357  -0.014 ± 0.017                              96.3     2 sur 5  ➖ dans le bruit
                             Meilleur contre meilleur (LogisticRegression → LogisticRegression)                              0.388                               0.364  -0.023 ± 0.020                              94.0     1 sur 5          ❌ perte

**Comparaison avec la variante complète, pli par pli : Précision moyenne**

Précision moyenne, complet  Précision moyenne, variante            Gain  Part du score complet gardée (%) Plis gagnés  Verdict
Variante                     Modèle                                                                                                                                                                            
PAY_n seuls + PAY_1 corrigée LogisticRegression                                                       0.504                        0.486  -0.018 ± 0.005                              96.4     0 sur 5  ❌ perte
                             MLPClassifier                                                            0.539                        0.521  -0.019 ± 0.008                              96.6     0 sur 5  ❌ perte
                             RandomForest                                                             0.555                        0.532  -0.023 ± 0.003                              95.8     0 sur 5  ❌ perte
                             CatBoost                                                                 0.556                        0.531  -0.025 ± 0.003                              95.6     0 sur 5  ❌ perte
                             Meilleur contre meilleur (CatBoost → RandomForest)                       0.556                        0.532  -0.024 ± 0.005                              95.7     0 sur 5  ❌ perte

**Comment lire** : comme dans `essai_jeu_de_base`. La comparaison « PAY_n seuls + PAY_1 corrigée » face à « Complet + PAY_1 corrigée » dit ce que les montants apportent en plus des codifications.

In [10]:
# Enregistrement, client par client, des probabilités hors pli et des décisions au seuil optimisé (règle marginale),
# pour chaque variante et chaque modèle, dans le dossier local sorties/. Aucun réentraînement :
# les décisions sont celles des modèles de chaque pli, gardés en mémoire à la section 4
NOM_ESSAI = "jeu_de_base_pay1_corrigee"
dossier_sortie = Path.cwd() / "sorties"
dossier_sortie.mkdir(parents=True, exist_ok=True)
sortie = pd.DataFrame({"ID": original["ID"].to_numpy(), "dpnm": y})
for (variante, nom), p in probas.items():
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    decision = np.zeros(len(y), dtype=int)
    for m, (_, validation) in zip(modeles_plis[(variante, nom)], plis):
        decision[validation] = m.predict(X.iloc[validation])
    sortie[f"proba | {variante} | {nom}"] = p
    sortie[f"decision | {variante} | {nom}"] = decision
sortie.to_csv(dossier_sortie / f"{NOM_ESSAI}.csv", index=False)
print(f"Enregistré : {dossier_sortie / f'{NOM_ESSAI}.csv'} ({len(sortie)} clients, {len(probas)} couples variante × modèle)")

Enregistré : c:\Users\johan\VS Code Wild 2\PROJET_DEFAULT_CREDIT_CARD_ML\lab_ML\correction_codification_1\sorties\jeu_de_base_pay1_corrigee.csv (30000 clients, 8 couples variante × modèle)


## 5. Ce qui porte les prédictions : importance par permutation

**Principe** : sur chaque pli de validation, on reprend le modèle entraîné sur les 4 autres plis (section 4) et on **mélange** les valeurs d'une famille de colonnes entre les clients (les autres colonnes restent intactes). Le modèle perd ce que cette famille lui apportait : la **perte de précision moyenne** mesure son importance. La précision moyenne ne dépend d'aucun seuil : la mesure juge la capacité de détection, pas un point de fonctionnement, et elle est plus stable qu'un score pris en un seul point. Le mélange est répété 5 fois (graine fixée), puis la perte est moyennée sur les répétitions et sur les 5 plis. Une famille se mélange d'un bloc (les six mois ensemble, avec le même tirage pour chaque colonne), pour garder sa cohérence d'un mois à l'autre.

**Familles** : codifications (`PAY_1` à `PAY_6`), **`PAY_1_corrigee`** (à part, pour voir ce qu'elle apporte en plus), factures (`BILL_AMT1` à `6`), paiements (`PAY_AMT1` à `6`), plafond (`LIMIT_BAL`), démographie (`SEX`, `EDUCATION`, `MARRIAGE`, `AGE`). Le comportement du client, au sens des montants, ce sont les factures, les paiements et le plafond ; les codifications sont l'étiquette posée par la banque.

**Deux tableaux** :
- **par famille**, pour chaque variante et chaque modèle : perte de précision moyenne et son écart-type sur les plis, et la part de chaque famille dans la perte totale de la colonne ;
- **par colonne**, mélangée seule : le détail, pour voir quels mois et quels montants comptent.

**Ce que la permutation mesure** : ce que le modèle perd **quand la famille manque, les autres étant là**. Deux familles qui disent la même chose (les codifications et les paiements, par exemple) se remplacent l'une l'autre : chacune paraît alors peu importante, alors qu'ensemble elles portent beaucoup. C'est pourquoi cette section se lit avec la section 4 : les variantes « `PAY_n` seuls » et « Sans les `PAY_n` » disent ce que chaque bloc porte **seul**, la permutation dit ce qu'il apporte **en plus des autres**.

In [11]:
FAMILLES = {
    "Codifications (PAY_n)": PAY,
    "PAY_1 corrigée": ["PAY_1_corrigee"],
    "Factures (BILL_AMTn)": [f"BILL_AMT{n}" for n in range(1, 7)],
    "Paiements (PAY_AMTn)": [f"PAY_AMT{n}" for n in range(1, 7)],
    "Plafond (LIMIT_BAL)": ["LIMIT_BAL"],
    "Démographie": ["SEX", "EDUCATION", "MARRIAGE", "AGE"],
}
REPETITIONS = 5
generateur = np.random.default_rng(42)

def perte_par_permutation(modeles, X, colonnes):
    # Perte de précision moyenne sur chaque pli de validation quand les colonnes sont mélangées d'un bloc (même tirage pour toutes),
    # moyennée sur les répétitions
    pertes = []
    for m, (_, validation) in zip(modeles, plis):
        X_val, y_val = X.iloc[validation], y[validation]
        reference = average_precision_score(y_val, m.predict_proba(X_val)[:, 1])
        perte = []
        for _ in range(REPETITIONS):
            melange = X_val.copy()
            ordre_melange = generateur.permutation(len(melange))
            melange[colonnes] = melange[colonnes].to_numpy()[ordre_melange]
            perte.append(reference - average_precision_score(y_val, m.predict_proba(melange)[:, 1]))
        pertes.append(np.mean(perte))
    return pertes

importance_familles, importance_colonnes = {}, {}
for (variante, nom), modeles in modeles_plis.items():
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    for famille, colonnes in FAMILLES.items():
        presentes = [c for c in colonnes if c in X.columns]
        if presentes:
            importance_familles[(variante, nom, famille)] = perte_par_permutation(modeles, X, presentes)
    for colonne in X.columns:
        importance_colonnes[(variante, nom, colonne)] = perte_par_permutation(modeles, X, [colonne])
    print(f"{variante}, {nom} : fait")

Complet + PAY_1 corrigée, LogisticRegression : fait
Complet + PAY_1 corrigée, MLPClassifier : fait
Complet + PAY_1 corrigée, RandomForest : fait
Complet + PAY_1 corrigée, CatBoost : fait
PAY_n seuls + PAY_1 corrigée, LogisticRegression : fait
PAY_n seuls + PAY_1 corrigée, MLPClassifier : fait
PAY_n seuls + PAY_1 corrigée, RandomForest : fait
PAY_n seuls + PAY_1 corrigée, CatBoost : fait


In [12]:
# Par famille : une colonne par variante et par modèle, une ligne par famille
colonnes_tableau = list(dict.fromkeys((variante, nom) for variante, nom, _ in importance_familles))
perte = pd.DataFrame({cle: {famille: np.mean(importance_familles[(*cle, famille)])
                            for famille in FAMILLES if (*cle, famille) in importance_familles} for cle in colonnes_tableau})
ecart = pd.DataFrame({cle: {famille: np.std(importance_familles[(*cle, famille)])
                            for famille in FAMILLES if (*cle, famille) in importance_familles} for cle in colonnes_tableau})
part = perte.clip(lower=0) / perte.clip(lower=0).sum() * 100

display(Markdown("**Perte de précision moyenne quand la famille est mélangée** (moyenne ± écart-type sur les 5 plis ; « — » : famille absente de la variante)"))
texte = (perte.round(3).astype(str) + " ± " + ecart.round(3).astype(str)).where(perte.notna(), "—")
display(texte.rename_axis(index="Famille", columns=["Variante", "Modèle"]))
display(Markdown("**Part de chaque famille dans la perte totale** (%, pertes négatives comptées à zéro ; à lire colonne par colonne)"))
display(part.round(1).rename_axis(index="Famille", columns=["Variante", "Modèle"]))

**Perte de précision moyenne quand la famille est mélangée** (moyenne ± écart-type sur les 5 plis ; « — » : famille absente de la variante)

Variante              Complet + PAY_1 corrigée                                              PAY_n seuls + PAY_1 corrigée                                             
Modèle                      LogisticRegression  MLPClassifier   RandomForest       CatBoost           LogisticRegression  MLPClassifier   RandomForest       CatBoost
Famille                                                                                                                                                              
Codifications (PAY_n)            0.152 ± 0.009  0.236 ± 0.017  0.199 ± 0.003  0.201 ± 0.008                0.275 ± 0.008  0.281 ± 0.015   0.227 ± 0.01   0.25 ± 0.005
PAY_1 corrigée                   0.064 ± 0.008  0.018 ± 0.004  0.051 ± 0.004  0.027 ± 0.005               -0.006 ± 0.001  0.029 ± 0.003  0.038 ± 0.005  0.012 ± 0.005
Factures (BILL_AMTn)              0.01 ± 0.001  0.022 ± 0.002  0.011 ± 0.002  0.023 ± 0.004                            —              —              —              —
Paiements (PAY_AMTn)             0.058 ± 0.003  0.036 ± 0.005  0.028 ± 0.003  0.031 ± 0.004                            —              —              —              —
Plafond (LIMIT_BAL)              0.002 ± 0.001  0.009 ± 0.003  0.004 ± 0.001  0.014 ± 0.003                            —              —              —              —
Démographie                      0.004 ± 0.001  0.006 ± 0.001  0.002 ± 0.001  0.007 ± 0.001                            —              —              —              —

**Part de chaque famille dans la perte totale** (%, pertes négatives comptées à zéro ; à lire colonne par colonne)

Variante              Complet + PAY_1 corrigée                                     PAY_n seuls + PAY_1 corrigée                                    
Modèle                      LogisticRegression MLPClassifier RandomForest CatBoost           LogisticRegression MLPClassifier RandomForest CatBoost
Famille                                                                                                                                            
Codifications (PAY_n)                     52.5          72.3         67.8     66.6                        100.0          90.6         85.7     95.3
PAY_1 corrigée                            22.0           5.4         17.2      8.9                          0.0           9.4         14.3      4.7
Factures (BILL_AMTn)                       3.5           6.8          3.7      7.6                          NaN           NaN          NaN      NaN
Paiements (PAY_AMTn)                      20.1          10.9          9.4     10.2                          NaN           NaN          NaN      NaN
Plafond (LIMIT_BAL)                        0.7           2.7          1.3      4.6                          NaN           NaN          NaN      NaN
Démographie                                1.2           1.9          0.6      2.2                          NaN           NaN          NaN      NaN

In [13]:
# Par colonne, mélangée seule : perte de précision moyenne sur les 5 plis, triée sur la variante complète (moyenne des 4 modèles)
par_colonne = pd.DataFrame({(variante, nom): {colonne: np.mean(importance_colonnes[(variante, nom, colonne)])
                                              for v, n, colonne in importance_colonnes if (v, n) == (variante, nom)}
                            for variante, nom in colonnes_tableau})
ordre_colonnes = par_colonne["Complet + PAY_1 corrigée"].mean(axis=1).sort_values(ascending=False).index
with pd.option_context("display.max_rows", None):
    display(Markdown("**Perte de précision moyenne quand la colonne est mélangée seule** (moyenne sur les 5 plis ; vide : colonne absente de la variante)"))
    display(par_colonne.loc[ordre_colonnes].round(3).rename_axis(index="Colonne", columns=["Variante", "Modèle"]))

**Perte de précision moyenne quand la colonne est mélangée seule** (moyenne sur les 5 plis ; vide : colonne absente de la variante)

Variante       Complet + PAY_1 corrigée                                     PAY_n seuls + PAY_1 corrigée                                    
Modèle               LogisticRegression MLPClassifier RandomForest CatBoost           LogisticRegression MLPClassifier RandomForest CatBoost
Colonne                                                                                                                                     
PAY_1                             0.034         0.146        0.058    0.073                        0.228         0.183        0.059    0.107
PAY_1_corrigee                    0.064         0.017        0.049    0.028                       -0.005         0.029        0.037    0.012
BILL_AMT1                         0.006         0.010        0.003    0.013                          NaN           NaN          NaN      NaN
LIMIT_BAL                         0.002         0.009        0.004    0.013                          NaN           NaN          NaN      NaN
PAY_6                             0.007         0.009        0.004    0.005                       -0.007         0.010        0.003    0.010
PAY_3                             0.005         0.008        0.007    0.004                        0.003         0.012        0.006    0.008
PAY_5                             0.006         0.008        0.003    0.002                       -0.002         0.016        0.003    0.002
PAY_2                             0.002         0.005        0.007    0.004                        0.003         0.016        0.010    0.006
PAY_AMT2                          0.004         0.004        0.002    0.005                          NaN           NaN          NaN      NaN
PAY_AMT3                          0.002         0.007        0.002    0.004                          NaN           NaN          NaN      NaN
PAY_4                             0.003         0.002        0.006    0.003                       -0.003         0.010        0.005    0.004
PAY_AMT1                          0.003         0.006        0.002    0.002                          NaN           NaN          NaN      NaN
BILL_AMT5                        -0.000         0.008       -0.000    0.001                          NaN           NaN          NaN      NaN
BILL_AMT4                        -0.000         0.006        0.000    0.001                          NaN           NaN          NaN      NaN
BILL_AMT2                        -0.001         0.005        0.001    0.001                          NaN           NaN          NaN      NaN
BILL_AMT3                         0.001         0.003        0.001    0.003                          NaN           NaN          NaN      NaN
PAY_AMT4                          0.001         0.002        0.001    0.003                          NaN           NaN          NaN      NaN
EDUCATION                         0.000         0.002        0.000    0.003                          NaN           NaN          NaN      NaN
MARRIAGE                          0.001         0.001        0.000    0.002                          NaN           NaN          NaN      NaN
BILL_AMT6                         0.001         0.002        0.001    0.001                          NaN           NaN          NaN      NaN
AGE                               0.001         0.002        0.001    0.001                          NaN           NaN          NaN      NaN
PAY_AMT5                          0.000         0.003        0.000    0.000                          NaN           NaN          NaN      NaN
PAY_AMT6                         -0.000         0.001        0.001    0.002                          NaN           NaN          NaN      NaN
SEX                               0.000         0.001       -0.000    0.000                          NaN           NaN          NaN      NaN

**Comment lire** :
- **Par famille** : la ligne « PAY_1 corrigée » dit ce que la colonne apporte **en plus** de `PAY_1` et des autres variables. Les deux colonnes sont identiques pour 88 % des clients : elles se remplacent l'une l'autre, et la permutation sous-estime chacune (mélanger l'une laisse l'autre intacte).
- **Par colonne** : comparer la perte de `PAY_1` et celle de `PAY_1_corrigee` montre **laquelle les modèles regardent**. Si `PAY_1_corrigee` perd plus que `PAY_1`, les modèles préfèrent la version corrigée ; si `PAY_1` perd plus, le 1 d'origine leur dit quelque chose que la correction efface.

## 6. Comparaison avec `essai_jeu_de_base` : précision à rappel égal, pli par pli

Chaque variante de cet essai face à son équivalent d'`essai_jeu_de_base` (`data/ML/comparatif_global/jeu_de_base.csv`) : « Complet + PAY_1 corrigée » face à « Complet », « PAY_n seuls + PAY_1 corrigée » face à « PAY_n seuls ». Mêmes clients, même ordre, mêmes 5 plis. Les différences : `PAY_1_corrigee` en plus, et les codifications 1 d'avril à août corrigées (28 clients).

**Mesures, sur chaque pli** : précision à rappel égal (50, 60, 70, 80 %), précision moyenne, et à la règle marginale (rappel et précision au seuil optimisé de chaque essai). Verdict : « gain réel » si le gain moyen dépasse l'écart-type des gains sur les plis, « perte » s'il est en dessous de moins cet écart-type, sinon « dans le bruit ».

In [14]:
reference = pd.read_csv(root_dir / "data" / "ML" / "comparatif_global" / "jeu_de_base.csv")
assert (reference["ID"].to_numpy() == original["ID"].to_numpy()).all(), "clients pas dans le même ordre"
assert (reference["dpnm"].to_numpy() == y).all(), "cible différente"

EQUIVALENTS = {"Complet + PAY_1 corrigée": "Complet", "PAY_n seuls + PAY_1 corrigée": "PAY_n seuls"}
RAPPELS = [0.50, 0.60, 0.70, 0.80]

def precision_au_rappel(y_vrai, proba, rappel):
    return y_vrai[proba >= seuil_pour_rappel(y_vrai, proba, rappel)].mean()

def mesures_plis(proba, decision):
    m = {f"Précision au rappel de {r:.0%}": [] for r in RAPPELS}
    m.update({"Précision moyenne": [], "Rappel, règle marginale": [], "Précision, règle marginale": []})
    for _, validation in plis:
        y_val, p_val, d_val = y[validation], proba[validation], decision[validation].astype(bool)
        for r in RAPPELS:
            m[f"Précision au rappel de {r:.0%}"].append(precision_au_rappel(y_val, p_val, r))
        m["Précision moyenne"].append(average_precision_score(y_val, p_val))
        m["Rappel, règle marginale"].append(recall_score(y_val, d_val))
        m["Précision, règle marginale"].append(precision_score(y_val, d_val, zero_division=0))
    return {k: np.array(v) for k, v in m.items()}

comparaison = {}
for (variante, nom), p in probas.items():
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    d = np.zeros(len(y), dtype=int)
    for m, (_, validation) in zip(modeles_plis[(variante, nom)], plis):
        d[validation] = m.predict(X.iloc[validation])
    equivalent = EQUIVALENTS[variante]
    comparaison[(variante, nom)] = {
        "jeu de base": mesures_plis(reference[f"proba | {equivalent} | {nom}"].to_numpy(), reference[f"decision | {equivalent} | {nom}"].to_numpy()),
        "avec PAY_1 corrigée": mesures_plis(p, d),
    }
MESURES_COMPAREES = list(next(iter(comparaison.values()))["avec PAY_1 corrigée"])

def ligne_comparaison(variante, libelle, ref, comp, mesure):
    gains = comp[mesure] - ref[mesure]
    return {"Variante": variante, "Modèle": libelle, "Mesure": mesure,
            "Jeu de base (%)": round(ref[mesure].mean() * 100, 1), "Avec PAY_1 corrigée (%)": round(comp[mesure].mean() * 100, 1),
            "Gain (pts)": f"{gains.mean() * 100:+.2f} ± {gains.std() * 100:.2f}",
            "Plis gagnés": f"{(gains > 0).sum()} sur 5", "Verdict": verdict(gains)}

lignes = []
for variante in VARIANTES:
    for nom in MODELES_A_TESTER:
        c = comparaison[(variante, nom)]
        for mesure in MESURES_COMPAREES:
            lignes.append(ligne_comparaison(variante, nom, c["jeu de base"], c["avec PAY_1 corrigée"], mesure))
    meilleur_ref = max(MODELES_A_TESTER, key=lambda n: comparaison[(variante, n)]["jeu de base"]["Précision moyenne"].mean())
    meilleur_comp = max(MODELES_A_TESTER, key=lambda n: comparaison[(variante, n)]["avec PAY_1 corrigée"]["Précision moyenne"].mean())
    for mesure in MESURES_COMPAREES:
        lignes.append(ligne_comparaison(variante, f"Meilleur contre meilleur ({meilleur_ref} → {meilleur_comp})",
                                        comparaison[(variante, meilleur_ref)]["jeu de base"], comparaison[(variante, meilleur_comp)]["avec PAY_1 corrigée"], mesure))
table = pd.DataFrame(lignes)

display(Markdown("**Synthèse : gain avec PAY_1 corrigée, en points, par mesure** (moyenne sur les 5 plis ± écart-type ; une colonne par modèle)"))
display(table.pivot_table(index=["Variante", "Mesure"], columns="Modèle", values="Gain (pts)", aggfunc="first", sort=False))
display(Markdown("**Verdicts, par mesure**"))
display(table.pivot_table(index=["Variante", "Mesure"], columns="Modèle", values="Verdict", aggfunc="first", sort=False))
with pd.option_context("display.max_rows", None):
    display(Markdown("**Détail, pli par pli**"))
    display(table.set_index(["Variante", "Modèle", "Mesure"]))

**Synthèse : gain avec PAY_1 corrigée, en points, par mesure** (moyenne sur les 5 plis ± écart-type ; une colonne par modèle)

Modèle                                                  LogisticRegression MLPClassifier  RandomForest      CatBoost Meilleur contre meilleur (CatBoost → CatBoost) Meilleur contre meilleur (RandomForest → RandomForest)
Variante                     Mesure                                                                                                                                                                                       
Complet + PAY_1 corrigée     Précision au rappel de 50%       +1.20 ± 0.34  -0.93 ± 1.01  -0.38 ± 0.50  -0.29 ± 0.27                                   -0.29 ± 0.27                                                    NaN
                             Précision au rappel de 60%       +0.34 ± 0.92  -0.17 ± 0.53  -0.55 ± 0.42  +0.03 ± 0.57                                   +0.03 ± 0.57                                                    NaN
                             Précision au rappel de 70%       +0.69 ± 0.46  -0.05 ± 0.71  -0.27 ± 0.25  -0.26 ± 0.46                                   -0.26 ± 0.46                                                    NaN
                             Précision au rappel de 80%       +0.68 ± 0.12  +0.14 ± 0.41  -0.58 ± 0.47  +0.36 ± 0.30                                   +0.36 ± 0.30                                                    NaN
                             Précision moyenne                +0.46 ± 0.16  -0.08 ± 0.41  +0.37 ± 0.25  -0.06 ± 0.30                                   -0.06 ± 0.30                                                    NaN
                             Rappel, règle marginale          +8.79 ± 2.08  +0.33 ± 3.35  -1.90 ± 3.08  -0.21 ± 0.76                                   -0.21 ± 0.76                                                    NaN
                             Précision, règle marginale       -1.82 ± 0.86  -0.17 ± 1.18  +3.54 ± 2.72  -0.12 ± 0.71                                   -0.12 ± 0.71                                                    NaN
PAY_n seuls + PAY_1 corrigée Précision au rappel de 50%       -0.40 ± 0.34  -0.88 ± 0.58  +0.42 ± 0.67  -0.42 ± 1.15                                            NaN                                           +0.42 ± 0.67
                             Précision au rappel de 60%       -0.31 ± 0.58  +0.37 ± 1.85  +0.09 ± 0.40  +0.56 ± 0.52                                            NaN                                           +0.09 ± 0.40
                             Précision au rappel de 70%       -0.04 ± 0.06  +0.08 ± 0.68  +0.53 ± 1.09  -0.39 ± 0.44                                            NaN                                           +0.53 ± 1.09
                             Précision au rappel de 80%       -0.05 ± 0.05  -0.15 ± 0.74  -0.22 ± 0.49  -0.05 ± 0.26                                            NaN                                           -0.22 ± 0.49
                             Précision moyenne                -0.03 ± 0.13  +0.32 ± 0.55  +0.11 ± 0.06  +0.08 ± 0.23                                            NaN                                           +0.11 ± 0.06
                             Rappel, règle marginale          -1.36 ± 1.51  -0.14 ± 2.31  +0.42 ± 0.16  -0.57 ± 0.35                                            NaN                                           +0.42 ± 0.16
                             Précision, règle marginale       +0.41 ± 0.85  -0.15 ± 1.41  +0.05 ± 0.30  +0.69 ± 0.62                                            NaN                                           +0.05 ± 0.30

**Verdicts, par mesure**

Modèle                                                  LogisticRegression    MLPClassifier     RandomForest         CatBoost Meilleur contre meilleur (CatBoost → CatBoost) Meilleur contre meilleur (RandomForest → RandomForest)
Variante                     Mesure                                                                                                                                                                                                
Complet + PAY_1 corrigée     Précision au rappel de 50%        ✅ gain réel  ➖ dans le bruit  ➖ dans le bruit          ❌ perte                                        ❌ perte                                                    NaN
                             Précision au rappel de 60%    ➖ dans le bruit  ➖ dans le bruit          ❌ perte  ➖ dans le bruit                                ➖ dans le bruit                                                    NaN
                             Précision au rappel de 70%        ✅ gain réel  ➖ dans le bruit          ❌ perte  ➖ dans le bruit                                ➖ dans le bruit                                                    NaN
                             Précision au rappel de 80%        ✅ gain réel  ➖ dans le bruit          ❌ perte      ✅ gain réel                                    ✅ gain réel                                                    NaN
                             Précision moyenne                 ✅ gain réel  ➖ dans le bruit      ✅ gain réel  ➖ dans le bruit                                ➖ dans le bruit                                                    NaN
                             Rappel, règle marginale           ✅ gain réel  ➖ dans le bruit  ➖ dans le bruit  ➖ dans le bruit                                ➖ dans le bruit                                                    NaN
                             Précision, règle marginale            ❌ perte  ➖ dans le bruit      ✅ gain réel  ➖ dans le bruit                                ➖ dans le bruit                                                    NaN
PAY_n seuls + PAY_1 corrigée Précision au rappel de 50%            ❌ perte          ❌ perte  ➖ dans le bruit  ➖ dans le bruit                                            NaN                                        ➖ dans le bruit
                             Précision au rappel de 60%    ➖ dans le bruit  ➖ dans le bruit  ➖ dans le bruit      ✅ gain réel                                            NaN                                        ➖ dans le bruit
                             Précision au rappel de 70%    ➖ dans le bruit  ➖ dans le bruit  ➖ dans le bruit  ➖ dans le bruit                                            NaN                                        ➖ dans le bruit
                             Précision au rappel de 80%    ➖ dans le bruit  ➖ dans le bruit  ➖ dans le bruit  ➖ dans le bruit                                            NaN                                        ➖ dans le bruit
                             Précision moyenne             ➖ dans le bruit  ➖ dans le bruit      ✅ gain réel  ➖ dans le bruit                                            NaN                                            ✅ gain réel
                             Rappel, règle marginale       ➖ dans le bruit  ➖ dans le bruit      ✅ gain réel          ❌ perte                                            NaN                                            ✅ gain réel
                             Précision, règle marginale    ➖ dans le bruit  ➖ dans le bruit  ➖ dans le bruit      ✅ gain réel                                            NaN                                        ➖ dans le bruit

**Détail, pli par pli**

Jeu de base (%)  Avec PAY_1 corrigée (%)    Gain (pts) Plis gagnés          Verdict
Variante                     Modèle                                                 Mesure                                                                                                         
Complet + PAY_1 corrigée     LogisticRegression                                     Précision au rappel de 50%             52.5                     53.7  +1.20 ± 0.34     5 sur 5      ✅ gain réel
                                                                                    Précision au rappel de 60%             45.3                     45.6  +0.34 ± 0.92     3 sur 5  ➖ dans le bruit
                                                                                    Précision au rappel de 70%             37.5                     38.2  +0.69 ± 0.46     4 sur 5      ✅ gain réel
                                                                                    Précision au rappel de 80%             31.4                     32.1  +0.68 ± 0.12     5 sur 5      ✅ gain réel
                                                                                    Précision moyenne                      49.9                     50.4  +0.46 ± 0.16     5 sur 5      ✅ gain réel
                                                                                    Rappel, règle marginale                30.0                     38.8  +8.79 ± 2.08     5 sur 5      ✅ gain réel
                                                                                    Précision, règle marginale             62.4                     60.6  -1.82 ± 0.86     0 sur 5          ❌ perte
                             MLPClassifier                                          Précision au rappel de 50%             57.7                     56.8  -0.93 ± 1.01     1 sur 5  ➖ dans le bruit
                                                                                    Précision au rappel de 60%             48.8                     48.6  -0.17 ± 0.53     2 sur 5  ➖ dans le bruit
                                                                                    Précision au rappel de 70%             40.7                     40.7  -0.05 ± 0.71     2 sur 5  ➖ dans le bruit
                                                                                    Précision au rappel de 80%             33.7                     33.8  +0.14 ± 0.41     4 sur 5  ➖ dans le bruit
                                                                                    Précision moyenne                      54.0                     53.9  -0.08 ± 0.41     1 sur 5  ➖ dans le bruit
                                                                                    Rappel, règle marginale                35.5                     35.8  +0.33 ± 3.35     4 sur 5  ➖ dans le bruit
                                                                                    Précision, règle marginale             66.4                     66.3  -0.17 ± 1.18     2 sur 5  ➖ dans le bruit
                             RandomForest                                           Précision au rappel de 50%             57.4                     57.1  -0.38 ± 0.50     2 sur 5  ➖ dans le bruit
                                                                                    Précision au rappel de 60%             49.6                     49.1  -0.55 ± 0.42     1 sur 5          ❌ perte
                                                                                    Précision au rappel de 70%             41.5                     41.2  -0.27 ± 0.25     1 sur 5          ❌ perte
                                                                                    Précision au rappel de 80%             34.4                     33.8  -0.58 ± 0.47     1 sur 5          ❌ perte
                                                                                    Précision moyenne                      55.1                     55.5  +0.37 ± 0.25     4 sur 5      ✅ gain r

**Comment lire** :
- **Précision à rappel égal** : à chaque ligne, les deux essais détectent la même part des défauts ; un gain positif veut dire que, pour détecter autant de défauts, la version avec `PAY_1_corrigee` signale moins de bons clients.
- **Règle marginale** : rappel et précision se lisent ensemble.
- Le résultat se lit avec la section 5 : un gain dans le bruit avec une forte importance de `PAY_1_corrigee` voudrait dire que les modèles la regardent à la place de `PAY_1`, sans y gagner.

## 7. Conclusion

*À écrire par l'utilisateur après lecture des résultats.*